# EfficientGPT: reproduce the Qwen3-1.7B decode benchmark on Colab

Runs `research/release/reproduce.sh --quick` (EfficientGPT int8 + CUDA-graph decoder vs Hugging Face bf16) on a
Colab GPU and packs the raw results into one archive you can download and post under
**Discussions → Hardware results** on https://github.com/BagelsFun/efficientgpt.

**GPU:** compute capability 8.0+ and 8 GB+. On Colab that means an **L4** (sm_89) or **A100** (sm_80) runtime; the
**T4 is not supported** (sm_75: the int8 kernel uses bf16 tensor-core `tl.dot`). Which GPUs you get depends on your
Colab plan.

**Tested once on Colab** by the author (NVIDIA L4, 2026-11-01, `reports/L4_LINUX_VALIDATION_2026-11-01.md`): the
code of steps 1–4 ran there as it is here; the benchmark ran in the background as step 5 now does. Two things are
newer than that run: steps 5–6 in their current form (background run, progress snapshots), tested only outside
Colab, and the pinned commit, which adds one privacy-filter fix to the code that ran there. Colab's Python is 3.13
(the published benchmarks used 3.11). Please report whatever happens, including failures.

**Privacy:** the script removes GPU serial numbers, UUIDs and other per-device IDs, the list of running programs and paths under the home
directory (`/root` on Colab). Other paths (e.g. `/content/...`) and text remain: look through the archive before
posting it.

**Keep the computer running this browser awake** for the whole run: if the browser sleeps, Colab can replace the
runtime and every file on it is lost (this happened in the first L4 attempt).

Runtime: not measured on Colab. Expect several GB of downloads (PyTorch, the model) plus the benchmark itself.
Runtime → Change runtime type → GPU (L4 or A100), then Runtime → Run all. Step 5 starts the benchmark in the
background; re-run step 6 every few minutes to see progress and download results so far.

In [ ]:
# Settings
import os
import subprocess
import sys

REPO_URL = 'https://github.com/BagelsFun/efficientgpt'
COMMIT = '718e3b36e27f1f0b523326ecd3fc98fd84783eee'  # the L4 run's code (bb1a5bf) plus a privacy-filter fix
ARGS = ['--quick']  # use [] for the full run with torchao (llama.cpp is skipped on Colab: no Ollama)
LOCAL_REPO = os.environ.get('EFFICIENTGPT_LOCAL_REPO')  # set only when testing outside Colab
if hasattr(sys.stdout, 'reconfigure'):  # outside Colab a redirected stdout may not be UTF-8 (the report contains Δ, ×)
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
if LOCAL_REPO:
    ARGS = os.environ.get('EFFICIENTGPT_ARGS', '--dry-run').split()


def run(cmd, **kw):
    print('$', ' '.join(cmd), flush=True)
    return subprocess.run(cmd, check=True, **kw)

In [ ]:
# Step 1: check the GPU before installing anything
NO_GPU = 'No NVIDIA GPU found. In Colab: Runtime -> Change runtime type -> GPU (L4 or A100).'


def gpu_info():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap,memory.total,driver_version',
                              '--format=csv,noheader,nounits'], capture_output=True, text=True)
    except FileNotFoundError:  # CPU-only runtime: no nvidia-smi at all
        raise RuntimeError(NO_GPU) from None
    if out.returncode != 0 or not out.stdout.strip():
        raise RuntimeError(NO_GPU)
    name, cap, mem, driver = [x.strip() for x in out.stdout.strip().splitlines()[0].split(',')]
    try:
        return name, float(cap), float(mem), driver
    except ValueError:  # e.g. "[N/A]" from an old driver
        raise RuntimeError(f'Could not read compute capability / memory for {name} ({cap}, {mem}).') from None


name, cap, mem_mib, driver = gpu_info()
print(f'GPU: {name}, compute capability {cap}, {mem_mib:.0f} MiB, driver {driver}')
if cap < 8.0:
    raise RuntimeError(f'{name} has compute capability {cap}; EfficientGPT needs 8.0 or newer (e.g. L4 or A100). '
                       'The T4 is not supported.')
if mem_mib < 7900:
    raise RuntimeError(f'{name} has {mem_mib:.0f} MiB; 8 GB or more is needed.')

In [ ]:
# Step 2: install the versions the benchmark was measured with
if not LOCAL_REPO:
    # Colab's torchvision / torchaudio are built for its own torch and can break imports after the torch change.
    # No restart is needed: every later step runs in a fresh process.
    run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchvision', 'torchaudio'])
    # '+cu128' forces the build the benchmark used: Colab preinstalls torch 2.11.0+cu130, which a plain '==2.11.0' accepts
    run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.11.0+cu128', '--index-url',
         'https://download.pytorch.org/whl/cu128'])
    run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.21.0', 'torchao==0.18.0', 'numpy', 'tiktoken'])
else:
    print('local test: skipping pip installs')

In [ ]:
# Step 3: get the code at a fixed version
if LOCAL_REPO:
    repo = LOCAL_REPO
    print('local test: using', repo)
else:
    repo = '/content/efficientgpt'
    if not os.path.isdir(repo):
        run(['git', 'clone', '-q', REPO_URL, repo])
    run(['git', '-C', repo, 'checkout', '-q', COMMIT])
print(subprocess.run(['git', '-C', repo, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())

In [ ]:
# Step 4: check that PyTorch sees the GPU (in a fresh process, so a pre-imported torch can't interfere)
check = subprocess.run([sys.executable, '-c', 'import torch, triton; print(torch.__version__, triton.__version__, '
                        'torch.cuda.is_available(), torch.cuda.get_device_capability() if torch.cuda.is_available() else None)'],
                       capture_output=True, text=True)
print(check.stdout.strip() or check.stderr.strip()[-2000:])
if check.returncode != 0 or ' True ' not in f' {check.stdout} ':
    raise RuntimeError('PyTorch cannot use the GPU. The driver may be too old for CUDA 12.8; please report the '
                       'driver version printed in step 1.')

In [ ]:
# Step 5: start the benchmark in the background
# It runs as its own process and logs to a file, so it keeps going if this tab closes, as long as Colab keeps the runtime.
if globals().get('proc') is not None and proc.poll() is None:  # don't start a second run that would share the GPU
    raise RuntimeError(f'A benchmark is already running (process {proc.pid}); use step 6 to follow it.')
env = dict(os.environ, PYTHON=sys.executable)
LOG = os.path.join(os.path.dirname(os.path.abspath(repo)), 'efficientgpt_console.log')  # outside the checkout
print('$ bash research/release/reproduce.sh', ' '.join(ARGS), '>', LOG, flush=True)
proc = subprocess.Popen(['bash', 'research/release/reproduce.sh', *ARGS], cwd=repo, env=env,
                        stdout=open(LOG, 'w', encoding='utf-8'), stderr=subprocess.STDOUT)
print('Started (process', proc.pid, '). Run the next cell every few minutes: it shows progress and downloads the '
      'results so far, so an interrupted runtime does not lose everything.')

In [ ]:
# Step 6: progress, and save the results so far (run as often as you like)
import re
import tarfile
import time


def download(path):
    try:
        from google.colab import files  # only inside Colab
        files.download(path)
        print('Downloading', os.path.basename(path), '(if nothing starts: Files panel on the left; Colab files are '
              'deleted when the runtime ends)')
    except ImportError:
        print('(not in Colab: download skipped)', path)


text = open(LOG, encoding='utf-8', errors='replace').read()
print('\n'.join(text.splitlines()[-12:]))
proc_ = globals().get('proc')
if LOCAL_REPO and os.environ.get('EFFICIENTGPT_WAIT') and proc_ is not None:  # local testing only
    proc_.wait()
    text = open(LOG, encoding='utf-8', errors='replace').read()
m = re.search(r'^output: (\S+)', text, re.M)
out_dir = os.environ.get('EFFICIENTGPT_RESULTS_DIR') or (os.path.join(repo, m.group(1)) if m else None)
final_archive = out_dir.rstrip('/\\') + '.tar.gz' if out_dir else None
if proc_ is not None:
    running, rc = proc_.poll() is None, proc_.returncode
    if running:
        print('\nStatus: still running.')
    else:
        note = {0: '', 1: ' (some steps failed; the archive still has everything that ran)',
                2: ' (the script stopped before running; no archive was made)'}.get(
            rc, ' (the process was stopped or crashed; the archive may be incomplete or missing)')
        print(f'\nStatus: finished with exit status {rc}{note}')
else:  # e.g. after a kernel restart: the notebook no longer knows the process, so judge from the files
    running = not (final_archive and os.path.exists(final_archive))
    print('\nStatus: ' + ('unknown (the notebook lost track of the process, e.g. after a restart); if the log above '
                         'still grows when you re-run this cell, it is still running.' if running else
                         'finished (the final archive exists).'))
if not out_dir or not os.path.isdir(out_dir):
    print('No results directory yet (a --dry-run writes none).')
elif running:  # partial snapshot; home-directory paths are scrubbed only in the final archive
    snap = os.path.join(os.path.dirname(out_dir.rstrip('/\\')), f'partial_{time.strftime("%H%M%S")}.tar.gz')
    with tarfile.open(snap, 'w:gz') as t:
        t.add(out_dir, arcname=os.path.basename(out_dir.rstrip('/\\')))
        t.add(LOG, arcname='reproduce_console.log')
    print('Partial results (not yet privacy-filtered; for your own backup):')
    download(snap)
else:
    for name_ in ('STATUS.txt', os.path.join('bench', 'REPORT.md')):
        path = os.path.join(out_dir, name_)
        if os.path.exists(path):
            print(f'--- {name_}\n' + open(path, encoding='utf-8').read())
    print('Final archive:', final_archive)
    print('Before posting: open it and look through every file, logs included, for user names, host names or paths.')
    if os.path.exists(final_archive):
        download(final_archive)